# 01 — Trovare le testate "parenti" (nuovo dataset)

Primo controllo sul nuovo dataset (post-scraping, 54.532 articoli, 354 testate), prima di qualsiasi classificatore.
Stesso metodo degli esperimenti preliminari, cieco alle etichette:

- articoli quasi uguali tra testate diverse (MinHash, pezzetti di 5 parole, Jaccard ≥ 0,5);
- due testate collegate se hanno ≥ 2 articoli quasi uguali in comune;
- gruppi = testate collegate anche indirettamente;
- verifica finale con il proprietario NewsGuard (non usato per costruire i gruppi).

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
from datasketch import MinHash, MinHashLSH
from comune import carica_dataset, normalizza, NEWSGUARD, DERIVED, RESULTS

df = carica_dataset().drop(columns=["label", "score"])   # niente etichette qui
df["norm"] = df.text.map(normalizza)
print(len(df), "articoli,", df.domain.nunique(), "testate")

54532 articoli, 354 testate


## 1. Articoli identici su testate diverse

In [2]:
testate_per_testo = df.groupby("norm").domain.nunique()
identici = df[df.norm.map(testate_per_testo) >= 2]
print("articoli con un testo identico su un'altra testata:", len(identici), "— testate:", identici.domain.nunique())

articoli con un testo identico su un'altra testata: 7392 — testate: 84


## 2. Articoli quasi uguali (MinHash)

In [3]:
def pezzetti(testo_norm, n=5):
    parole = testo_norm.split()
    return {" ".join(parole[i:i + n]) for i in range(max(1, len(parole) - n + 1))}

lsh = MinHashLSH(threshold=0.5, num_perm=128)
impronte = []
for i, testo in enumerate(df.norm):
    m = MinHash(num_perm=128, seed=1)
    m.update_batch([p.encode() for p in pezzetti(testo)])
    impronte.append(m)
    lsh.insert(i, m)

testata = df.domain.values
coppie = []
for i, m in enumerate(impronte):
    for j in lsh.query(m):
        if j > i and testata[i] != testata[j] and m.jaccard(impronte[j]) >= 0.5:
            coppie.append((i, j, round(m.jaccard(impronte[j]), 3)))
coppie = pd.DataFrame(coppie, columns=["articolo_a", "articolo_b", "jaccard"])
coppie["testata_a"], coppie["testata_b"] = testata[coppie.articolo_a], testata[coppie.articolo_b]
quasi = set(coppie.articolo_a) | set(coppie.articolo_b)
print("coppie di articoli quasi uguali su testate diverse:", len(coppie))
print("articoli con una copia su un'altra testata:", len(quasi), f"({len(quasi) / len(df):.0%})")

coppie di articoli quasi uguali su testate diverse: 8390
articoli con una copia su un'altra testata: 8100 (15%)


## 3. Dai legami ai gruppi

In [4]:
def gruppi(min_articoli=2, min_jaccard=0.5):
    c = coppie[coppie.jaccard >= min_jaccard].groupby(["testata_a", "testata_b"]).size()
    G = nx.Graph(); G.add_nodes_from(df.domain.unique()); G.add_edges_from(c[c >= min_articoli].index)
    componenti = sorted(nx.connected_components(G), key=len, reverse=True)
    return pd.Series({t: k for k, comp in enumerate(componenti) for t in comp}, name="gruppo")

gruppo = gruppi()
dimensione = gruppo.map(gruppo.value_counts())
print("gruppi con almeno 2 testate:", (gruppo.value_counts() >= 2).sum())
print("testate in un gruppo:", (dimensione >= 2).sum(), "su", len(gruppo))
riassunto = pd.DataFrame({"testate": gruppo.value_counts()})
riassunto["esempi"] = [", ".join(sorted(gruppo[gruppo == g].index)[:3]) for g in riassunto.index]
riassunto[riassunto.testate >= 2]

gruppi con almeno 2 testate: 26
testate in un gruppo: 116 su 354


,testate,esempi
gruppo,,
0,39,"amnews.com, andalusiastarnews.com, atmoreadvan..."
1,12,"amgreatness.com, conservativedailynews.com, ge..."
2,11,"editorial.huffingtonpost.com, highline.huffing..."
3,5,"americanjournaldaily.com, gopdailybrief.com, p..."
4,4,"generaldispatch.whatfinger.com, money.whatfing..."
5,3,"analyzingamerica.org, breakingreports.org, sar..."
6,3,"farmvilleherald.com, kenbridgevictoriadispatch..."
7,3,"talkingpointsmemo.com, truth-out.org, truthout..."
8,2,"americanpress.com, thehayride.com"


## 4. Sensibilità alle soglie + verifica con il proprietario

In [5]:
ng = pd.read_csv(NEWSGUARD, low_memory=False).drop_duplicates("Domain").set_index("Domain")
proprietario = pd.Series({t: (ng.Owner.get(t) if isinstance(ng.Owner.get(t), str) and ng.Owner.get(t) != "Unknown" else "?" + t)
                          for t in gruppo.index})
print("testate che condividono il proprietario con almeno un'altra:", (proprietario.map(proprietario.value_counts()) >= 2).sum())

def prec_rich(g):
    p, r = [], []
    for t in g.index:
        sg = set(g.index[g == g[t]]); so = set(proprietario.index[proprietario == proprietario[t]])
        p.append(len(sg & so) / len(sg)); r.append(len(sg & so) / len(so))
    return np.mean(p), np.mean(r)

righe = []
for mj in [0.5, 0.7, 0.9]:
    for ma in [1, 2, 3, 5]:
        g = gruppi(ma, mj); dim = g.value_counts(); p, r = prec_rich(g)
        righe.append({"somiglianza min.": mj, "articoli in comune min.": ma, "gruppi (≥2)": (dim >= 2).sum(),
                      "testate nei gruppi": dim[dim >= 2].sum(), "gruppo più grande": dim.iloc[0],
                      "precisione": round(p, 3), "richiamo": round(r, 3)})
pd.DataFrame(righe)

testate che condividono il proprietario con almeno un'altra: 146


,somiglianza min.,articoli in comune min.,gruppi (≥2),testate nei gruppi,gruppo più grande,precisione,richiamo
0,0.5,1,24,158,103,0.703,0.799
1,0.5,2,26,116,39,0.873,0.762
2,0.5,3,29,107,21,0.907,0.724
3,0.5,5,30,89,7,0.960,0.705
4,0.7,1,28,153,40,0.774,0.771
5,0.7,2,25,112,36,0.877,0.754
6,0.7,3,28,101,19,0.918,0.719
7,0.7,5,30,88,7,0.960,0.704
8,0.9,1,26,134,38,0.818,0.761
9,0.9,2,29,104,19,0.926,0.723


## 5. Salvataggio

In [6]:
pd.DataFrame({"gruppo": gruppo, "testate_nel_gruppo": dimensione}).rename_axis("domain").to_csv(RESULTS + "01_gruppi_testate.csv")
coppie.to_csv(DERIVED + "01_coppie_quasi_uguali.csv.gz", index=False)
print("Salvato. Testate indipendenti:", (dimensione == 1).sum(), "| in un gruppo:", (dimensione >= 2).sum())

Salvato. Testate indipendenti: 238 | in un gruppo: 116
